<a href="https://colab.research.google.com/github/simichakravarty16/ds2002-fa26/blob/main/notebooks/04-json-apis/2026-10-02%20%E2%80%94%20Normalize%20and%20Join%20Nested%20JSON%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  
**Name:** Simi Chakravarty
---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-08','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** line item of an order. In orders_json, each order contains one line which simultaneously represents one or more items. Once orders_json is joined to to vendors_json by vendor_id, this will remain the same as each row of the dataframe will contain the item, quantity, order, and vendor_id.

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd

orders = pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'])

assert len(orders) == 4
assert orders['qty'].sum() == 7

orders.head()

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-08


**In this question, I was tasked with flattening orders_json and asserting that the expected number of rows and the quantity was correct. First, I imported the pandas library (import pandas as pd) so that I would be able to use the pd.json_normalize() function. In order to turn the nested JSON into a flattened data frame, I used the pd.json_normalize() function, telling it to use the orders_json JSON and then defining record_path = 'lines', so that each dictionary within lines would become one rows containing the item and quantity. I defined meta = ['order', 'vendor_id'] so that the data frame will include these columns. I needed to define these as they are outside of 'lines'. Finally, I needed to confirm that there were 4 rows and that quantity summed to 7, which I did using the two assertions: len(orders) calculated the number of rows of the orders dataframe and orders['qty'].sum() calculated the sum of all values in the qty column. Since I did not receive an AssertionError, I know that the data frame has the expected number of rows and total quantity. I also used orders.head() to take a quick look at the format of the data frame.**

### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
vendors = pd.json_normalize(vendors_json)

joined = orders.merge(vendors, how='left', on='vendor_id', indicator=True)

unmatched = joined[joined['_merge'] == 'left_only']

print("Unmatched Vendor:", unmatched['vendor_id'].iloc[0])
print("Units attached to Unmatched Vendor:", unmatched['qty'].sum())

Unmatched Vendor: V-08
Units attached to Unmatched Vendor: 3


**In this question, I wanted to flatten the vendors_json into a data frame and left join it to orders, which I made in the previous question. First, I used the pd.json_normalize() function to flatten vendors_json. I did not need to define the record_path or meta arguments because vendors_json was already flat and not nested like orders_json, so it would already produce vendor_id, name, and zone columns. To join it to the orders dataframe, I used the .merge() method on orders, making sure that orders was the left dataframe and vendors was left-joined based on vendor_id, which was the common column between the two. I needed to use a left join to keep every row from orders even if there was not a matching vendor. I needed to define the indicator argument as true to add the _merge column to the joined data frame, which indicates whether each row matched (both) or if there was a row found in orders but not in vendors (left_only). Using the _merge column, I could find the unmatched vendor by create a subset of the data entitled unmatched, which filtered the dataframe for rows where the _merge column had a left_only value. Then, I printed the vendor_id and total quantity of this subset, which showed that the unmatched vendor was V-08 with 3 units.**

### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
joined_df = joined.copy()

# Decided to label the unmatched vendor as 'Unknown Name'
# so that its quantity will still be included in the dataframe
joined_df['name'] = joined_df['name'].fillna('Unknown Name')

joined_df.groupby('name')['qty'].sum()

,qty
name,
Cav Merch,1
Hoos Burgers,3
Unknown Name,3


**First, I made a copy of the joined dataframe using the .copy() method and assigned it to joined_df. I did this so I would not modify the original joined dataframe. I decided that the unmatched vendor should appear as 'Unknown Name' so that its quantity would still be included in the data frame while giving a clear indication that the vendor name is unknown. Since the unmatched vendor (V-08) did not have a matching name in the vendors dataframe, it shows up as NaN in the joined_df dataframe. Because of this, I could use the .fillna('Unknown Name') method to replace any NaN value with Unknown Name. In order to find the total quantity by vendor name, I used .groupby('name') to group all rows according to vendor name, selected for the quantity column, and then used .sum()to find the total quantity within each group.**

### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:
joined_df['zone'] = joined_df['zone'].fillna('Unknown Zone')

total_zone = joined_df.groupby('zone')['qty'].sum()

print("Total units:", total_zone.sum())
print("Difference:", joined_df['qty'].sum() - total_zone.sum())
total_zone

Total units: 7
Difference: 0


,qty
zone,
A,4
Unknown Zone,3


**Zone A sold the most items with a quantity of 4. To find this, I first wanted to label the zone of the unmatched vendor (V-08) with 'Unknown Zone' so that I could still see its quantity. Since the unmatched vendor did not have a matching zone in the vendors dataframe, it shows up as NaN in the joined_df dataframe. Because of this, I could use the .fillna('Unknown Zone') method to replace any NaN value with 'Unknown Zone'. Then, I used .groupby('zone') to group all rows according to vendor zone, selected for the quantity column, and then used .sum()to find the total quantity within each group. I assigned this to total_zone. Next, I calculated the total number of rows by summing the quantity column using total_zone.sum() and calculated the difference by subtracting it from the sum of the quantity column of the joined_df dataframe. The difference here is 0 because I assigned the zone of the missing vendor (V-08) to 'Unknown Zone' in order to keep the quantity.**

### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [6]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

rows = []

for order in ragged_json:
    lines = order.get('lines', [])

    if lines:
        for line in lines:
            rows.append({
                'order': order['order'],
                'vendor_id': order['vendor_id'],
                'item': line.get('item'),
                'qty': line.get('qty')
            })
    else:
        rows.append({
            'order': order['order'],
            'vendor_id': order['vendor_id'],
            'item': None,
            'qty': None
        })

ragged = pd.DataFrame(rows)

ragged

,order,vendor_id,item,qty
0,4,V-01,Soda,2.0
1,5,V-10,None,NaN
2,6,V-01,Fries,NaN


**Since order 5 was missing the lines key, I could not use pd.json_normalize() like I did to flatten the other JSONs. First, I created an empty list called rows, as each dataframe row would be built inside this list. Next, I used a for loop (for order in ragged_json:) to look at each order. I used lines = order.get('lines', []) to see if lines exist and, if so, to return the value associated with lines. If lines doesn't exist, it should return an empty list. If lines does exist, for line in lines: makes sure the code runs through every line in the the order and adds a new row to the list using rows.append(). Since all of the orders in the ragged JSON have an order number and vendor_id, I can extract that directly, but not all of them have lines or qty. This is why I use line.get('item') and line.get('qty'), so if one of these is missing it returns None. This makes sure that missing quantities are defined as missing rather than as 0. The else statement handles the instances when there are no lines and directly indicates the item and qty as None. After the loop, I used pd.DataFrame(rows) to turn the appended rows list into a dataframe.**

### Q6 — Validate

In [7]:
assert len(orders) == 4
assert orders['qty'].sum() == 7
assert len(joined) == len(orders), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


**In this question, the code was already provided, but I needed to change lines to orders, since that is the name I previously defined the flattened orders_json dataframe as. All of the assertions ran which confirmed that orders had 4 rows, that there were 7 units in total, and that the length of the joined dataframe was equal to the orders dataframe.**

### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a) One row per line item is a useful shape because you can see all of the items that are included in each order, including their quantity, order, and vendor_id. You can see a lot of useful information and decipher the total quantities for each item. If you instead used one row per order, there would be a lot of items grouped in each row, which would make it much harder to see the quantities of the individual items.

b) In Q5, the difference between a missing quantity and a quantity of zero is that a missing quantity was not provided, whereas a zero quantity means the actual quantity was 0. A missing quantity could be any number, but we are not sure what it is because it was not provided in the dataframe. If I had conflated missing quantities with 0 quantities, I would incorrectly be using unknown quantities like none of the items had been sold, which would make all other calculations incorrect.